# `extract.ai` — 2026 Best Practices Demo

This notebook walks through every new feature added to `wrangles.extract.ai` in the 2026 update.

**Prerequisites**
```
pip install wrangles pandas
```
Set your OpenAI key before running:
```bash
export OPENAI_API_KEY=sk-...
```
or set it in the cell below.

In [ ]:
import os
import pandas as pd
import wrangles

# Set your key here if not already in the environment
# os.environ['OPENAI_API_KEY'] = 'sk-...'

API_KEY = os.environ['OPENAI_API_KEY']
print('wrangles version:', wrangles.__version__)
print('Key loaded:', API_KEY[:8] + '...')

---
## 0 — Baseline (unchanged API)

Everything that worked before still works — no changes required to existing code.

In [ ]:
data = [
    'wrench 25mm',
    '6m cable',
    'screwdriver 3mm',
]

results = wrangles.extract.ai(
    data,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'length': {
            'type': 'string',
            'description': 'Any length found in the text (e.g. 25mm, 6m)',
        }
    },
    retries=2,
)

pd.DataFrame({'input': data, **{k: [r[k] for r in results] for k in results[0]}})

---
## 1 — `instructions`: system-level guidance for every row

Use `instructions` to set rules that apply across the whole batch.  
Each string becomes a separate **system** message, so the model treats it as a hard constraint.

In [ ]:
results = wrangles.extract.ai(
    data,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'length': {
            'type': 'string',
            'description': 'Any length found in the text',
        }
    },
    instructions='Always return length values in UPPERCASE (e.g. 25MM, 6M).',
    retries=2,
)

pd.DataFrame({'input': data, **{k: [r[k] for r in results] for k in results[0]}})

In [ ]:
# Multiple instructions — each becomes its own system message
results = wrangles.extract.ai(
    data,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'length': {'type': 'string', 'description': 'Any length found in the text'},
        'tool':   {'type': 'string', 'description': 'The type of tool described'},
    },
    instructions=[
        'Return length values with their unit attached (e.g. 25mm, not 25).',
        'Return tool names in lowercase.',
    ],
    retries=2,
)

pd.DataFrame({'input': data, **{k: [r[k] for r in results] for k in results[0]}})

---
## 2 — `taxonomy`: constrain outputs to a controlled vocabulary

Pass a list or dict of valid values.  
The model is instructed to use **only** values from the taxonomy.

In [ ]:
reviews = [
    "Best product I've ever bought. Totally worth it!",
    "Arrived broken. Customer service was useless.",
    "Does the job. Nothing special.",
    "Absolutely love it, exceeded all my expectations.",
    "Waste of money. Would not recommend.",
]

results = wrangles.extract.ai(
    reviews,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'sentiment': {
            'type': 'string',
            'description': 'Overall sentiment of the review',
        }
    },
    taxonomy=['positive', 'negative', 'neutral'],
    retries=2,
)

pd.DataFrame({'review': reviews, 'sentiment': [r['sentiment'] for r in results]})

In [ ]:
# Hierarchical taxonomy — nested dict
products = [
    'Apple iPhone 15 Pro 256GB',
    'Nike Air Max 270 Running Shoes',
    'Sony WH-1000XM5 Wireless Headphones',
    'Levi\'s 501 Original Jeans',
    'Samsung 65" QLED Smart TV',
]

results = wrangles.extract.ai(
    products,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'category': {
            'type': 'string',
            'description': 'Product category from the taxonomy',
        }
    },
    taxonomy={
        'Electronics': ['Phones', 'Laptops', 'Headphones', 'TVs', 'Cameras'],
        'Clothing':    ['Shoes', 'Jeans', 'Shirts', 'Jackets'],
        'Home':        ['Furniture', 'Kitchen', 'Garden'],
    },
    retries=2,
)

pd.DataFrame({'product': products, 'category': [r['category'] for r in results]})

---
## 3 — `examples`: few-shot learning

Provide input/output pairs that demonstrate the expected behaviour.  
Each example is inserted as a proper OpenAI **tool-call message triple** so
the model sees them as completed extractions, not just text.

This is the most powerful way to improve accuracy for unusual formats or
domain-specific conventions.

In [ ]:
# Without examples — model must infer the format from the description alone
part_descriptions = [
    'M8x1.25 hex bolt, grade 8.8, zinc plated, 40mm',
    'M6x1.0 socket head cap screw, A2 stainless, 25mm',
    'M10x1.5 flange nut, grade 10, plain finish',
]

output_schema = {
    'thread_size': {'type': 'string', 'description': 'Thread size (e.g. M8x1.25)'},
    'material':    {'type': 'string', 'description': 'Material or finish'},
    'length_mm':   {'type': 'string', 'description': 'Length in mm if present, else empty string'},
}

results_no_examples = wrangles.extract.ai(
    part_descriptions,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output=output_schema,
    retries=2,
)

print('--- Without examples ---')
pd.DataFrame(results_no_examples)

In [ ]:
# With examples — model sees the exact format expected
results_with_examples = wrangles.extract.ai(
    part_descriptions,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output=output_schema,
    examples=[
        {
            'input': 'M4x0.7 pan head screw, black oxide, 12mm',
            'output': {'thread_size': 'M4x0.7', 'material': 'black oxide', 'length_mm': '12'},
        },
        {
            'input': 'M12x1.75 hex nut, hot-dip galvanised',
            'output': {'thread_size': 'M12x1.75', 'material': 'hot-dip galvanised', 'length_mm': ''},
        },
    ],
    retries=2,
)

print('--- With examples ---')
pd.DataFrame(results_with_examples)

---
## 4 — `max_completion_tokens`: control cost and response length

`max_completion_tokens` is the current OpenAI parameter (replacing the deprecated `max_tokens`).  
Setting it caps the output length per row, which prevents runaway costs on
open-ended outputs and keeps latency predictable.

In [ ]:
descriptions = [
    'Cordless drill, 18V, 2-speed, with LED light and magnetic bit holder',
    'Stainless steel mixing bowl set, 3-piece, dishwasher safe',
    'Adjustable standing desk, electric, 120x60cm, oak finish',
]

results = wrangles.extract.ai(
    descriptions,
    api_key=API_KEY,
    model='gpt-4o-mini',
    output={
        'one_line_summary': {
            'type': 'string',
            'description': 'A concise one-line product summary',
        }
    },
    max_completion_tokens=60,   # keep it tight
    retries=2,
)

pd.DataFrame({'description': descriptions, 'summary': [r['one_line_summary'] for r in results]})

---
## 5 — Reasoning models (o3-mini, o1, …)

OpenAI o-series models reject `temperature`, `top_p` etc. with a hard error.  
The guard in `extract.ai` **silently strips** those parameters so you can switch
models without changing your code.

In [ ]:
# temperature=0.2 is stripped automatically for o3-mini — no error raised
results = wrangles.extract.ai(
    data,
    api_key=API_KEY,
    model='o3-mini',
    temperature=0.2,        # would cause a hard API error without the guard
    output={
        'length': {
            'type': 'string',
            'description': 'Any length found in the text',
        }
    },
    retries=2,
)

pd.DataFrame({'input': data, 'length': [r['length'] for r in results]})

---
## 6 — Combining everything in a recipe

All parameters work in YAML recipes too.

In [ ]:
df = pd.DataFrame({
    'description': [
        'M8x1.25 hex bolt, grade 8.8, zinc plated, 40mm',
        'M6x1.0 socket cap screw, A2 stainless, 25mm',
        'M10x1.5 flange nut, grade 10, plain finish',
    ]
})

df_out = wrangles.recipe.run(
    """
    wrangles:
      - extract.ai:
          input: description
          api_key: ${OPENAI_API_KEY}
          model: gpt-4o-mini
          max_completion_tokens: 80
          retries: 2
          instructions: Return thread_size exactly as written (e.g. M8x1.25).
          taxonomy:
            - zinc plated
            - A2 stainless
            - hot-dip galvanised
            - black oxide
            - plain
          examples:
            - input: M4x0.7 pan head screw, black oxide, 12mm
              output:
                thread_size: M4x0.7
                finish: black oxide
                length_mm: "12"
          output:
            thread_size:
              type: string
              description: Thread size (e.g. M8x1.25)
            finish:
              type: string
              description: Surface finish or material from the taxonomy
            length_mm:
              type: string
              description: Length in mm, empty string if not present
    """,
    dataframe=df,
)

df_out

---
## 7 — Rate-limit-aware backoff (automatic, no API call needed)

This improvement is internal — no new parameter required.  
The cell below shows what the retry loop does differently.

In [ ]:
from unittest.mock import patch, MagicMock
import json as _json
import wrangles.openai as oai

def make_response(status, headers=None, body=None):
    r = MagicMock()
    r.ok = status < 400
    r.status_code = status
    r.headers = headers or {}
    r.json.return_value = body or {}
    return r

ok = make_response(200)
ok.json.return_value = {
    'choices': [{'message': {'tool_calls': [{'function': {'arguments': '{"length":"25mm"}'}}]}}]
}
rate_limited = make_response(
    429,
    headers={'retry-after': '7'},
    body={'error': {'message': 'Rate limit exceeded'}},
)

sleep_calls = []

with patch('wrangles.openai._requests.post', side_effect=[rate_limited, ok]), \
     patch('wrangles.openai._time.sleep', side_effect=lambda s: sleep_calls.append(s)):
    oai.chatGPT(
        'wrench 25mm',
        api_key='test_key',
        settings={
            'model': 'gpt-4o-mini',
            'messages': [],
            'tools': [{'type': 'function', 'function': {
                'name': 'parse_output',
                'parameters': {'required': ['length']}
            }}]
        },
        retries=1,
    )

print(f'429 received → slept exactly {sleep_calls[0]}s (from Retry-After header, not guessed)')

---
## Summary of new parameters

| Parameter | Type | Purpose |
|---|---|---|
| `instructions` | `str \| list[str]` | System-level rules applied to every row |
| `examples` | `list[dict]` | Few-shot input/output pairs for better accuracy |
| `taxonomy` | `dict \| list` | Controlled vocabulary the model must draw from |
| `max_completion_tokens` | `int` | Cap output length per row (cost + latency control) |

Internal improvements (no new parameters needed):

| Improvement | What changed |
|---|---|
| Reasoning-model guard | Strips `temperature` / `top_p` / etc. for o-series models automatically |
| Rate-limit-aware backoff | Reads `Retry-After` header on 429s instead of guessing |
| Centralised prompt builder | `wrangles.openai.build_extract_messages()` — reusable by other wrangles |